# Capital Asset Pricing Model (CAPM) Stock Analysis

This notebook implements the Capital Asset Pricing Model (CAPM) to evaluate risk and return profiles for selected individual stocks (AAPL, MSFT, GOOG, AMZN) relative to the market index (SPY).

## Theoretical Background
The CAPM model describes the relationship between systematic risk and expected return for assets, particularly stocks. The formula is:

$$E(R_i) = R_f + \beta_i (E(R_m) - R_f)$$

Where:
- $E(R_i)$ is the expected return of the asset.
- $R_f$ is the risk-free rate.
- $\beta_i$ is the beta of the asset (measure of systematic risk relative to the market).
- $E(R_m)$ is the expected return of the market index.
- $E(R_m) - R_f$ is the market risk premium.

We also compute key metrics:
- **Alpha ($\alpha$)**: The excess return of the stock relative to the return predicted by CAPM (measure of active return).
- **Sharpe Ratio**: Reward-to-volatility ratio, defined as $(R_i - R_f) / \sigma_i$.
- **Treynor Ratio**: Reward-to-beta ratio, defined as $(R_i - R_f) / \beta_i$.

In [ ]:
# Imports Section
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Set plots design
sns.set_theme(style="darkgrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

## 1. Data Ingestion

We fetch historical adjusted closing prices for our target stocks (Apple, Microsoft, Google, Amazon) and the benchmark index S&P 500 (SPY) over a 5-year period.

In [ ]:
# Define tickers and period
tickers = ["SPY", "AAPL", "MSFT", "GOOG", "AMZN"]
start_date = "2019-01-01"
end_date = "2023-12-31"

# Fetch data using yfinance
raw_data = yf.download(tickers, start=start_date, end=end_date)['Adj Close']

# Handle any missing values
data = raw_data.ffill().bfill()

# Display top records
data.head()

## 2. Returns Calculation

We compute daily returns for each asset. Daily return is calculated as: $R_{t} = \frac{P_t - P_{t-1}}{P_{t-1}}$.

In [ ]:
# Calculate daily returns
returns = data.pct_change().dropna()

# Show descriptive statistics
returns.describe()

## 3. Cumulative Growth Visualization

Let's visualize the growth of a hypothetical $1 investment in each asset starting in 2019.

In [ ]:
# Calculate cumulative growth
cumulative_growth = (1 + returns).cumprod()

# Plot growth
plt.figure(figsize=(14, 7))
for col in cumulative_growth.columns:
    width = 2.5 if col == 'SPY' else 1.5
    plt.plot(cumulative_growth.index, cumulative_growth[col], label=col, linewidth=width)

plt.title("Growth of a $1 Investment (2019 - 2023)", fontsize=16, weight='bold')
plt.xlabel("Date")
plt.ylabel("Portfolio Value ($)")
plt.legend(fontsize=12)
plt.show()

## 4. CAPM Regression (Alpha & Beta Estimation)

We run ordinary least squares (OLS) regressions of each stock's excess return against the market excess return (SPY). 

We assume an annualized risk-free rate of **4.0%** ($R_f = 0.04$). The daily risk-free rate is calculated as: $R_{f, \text{daily}} = \frac{R_f}{252}$.

In [ ]:
# Define annual risk-free rate and daily conversion
rf_annual = 0.04
rf_daily = rf_annual / 252

# Calculate excess returns
excess_returns = returns.copy()
for col in excess_returns.columns:
    excess_returns[col] = excess_returns[col] - rf_daily

# Define market excess returns (independent variable)
X = excess_returns['SPY']
X_with_const = sm.add_constant(X)

# Regression container
results = {}

# Loop through stocks and fit OLS
stocks = [t for t in tickers if t != 'SPY']
for stock in stocks:
    Y = excess_returns[stock]
    model = sm.OLS(Y, X_with_const).fit()
    results[stock] = {
        "alpha_daily": model.params['const'],
        "beta": model.params['SPY'],
        "r_squared": model.rsquared,
        "p_values": model.pvalues
    }
    print(f"=== Regression Summary: {stock} ===")
    print(f"Alpha (Daily): {model.params['const']:.6f}")
    print(f"Beta:          {model.params['SPY']:.4f}")
    print(f"R-Squared:     {model.rsquared:.4f}\n")

## 5. Risk-Adjusted Ratios (Sharpe & Treynor)

We calculate:
1. **Annualized Volatility**: $\sigma_{\text{annual}} = \sigma_{\text{daily}} \times \sqrt{252}$
2. **Annualized Returns**: $R_{\text{annual}} = R_{\text{daily}} \times 252$
3. **Sharpe Ratio**: $\frac{R_{\text{annual}} - R_{f}}{\sigma_{\text{annual}}}$
4. **Treynor Ratio**: $\frac{R_{\text{annual}} - R_{f}}{\beta}$

In [ ]:
summary_data = []

for stock in stocks:
    # Returns and volatility
    stock_returns = returns[stock]
    ann_return = stock_returns.mean() * 252
    ann_vol = stock_returns.std() * np.sqrt(252)
    
    # Alpha/Beta from regression
    beta = results[stock]["beta"]
    alpha_ann = results[stock]["alpha_daily"] * 252 # Annualized alpha
    
    # Ratios
    sharpe = (ann_return - rf_annual) / ann_vol if ann_vol > 0 else 0
    treynor = (ann_return - rf_annual) / beta if beta > 0 else 0
    
    summary_data.append({
        "Stock": stock,
        "Annualized Return": f"{ann_return*100:.2f}%",
        "Annualized Volatility": f"{ann_vol*100:.2f}%",
        "Beta (Systematic Risk)": round(beta, 2),
        "Annualized Alpha": f"{alpha_ann*100:.2f}%",
        "Sharpe Ratio": round(sharpe, 3),
        "Treynor Ratio": round(treynor, 3)
    })

# Add Market benchmark SPY
spy_returns = returns['SPY']
spy_ann_return = spy_returns.mean() * 252
spy_ann_vol = spy_returns.std() * np.sqrt(252)
spy_sharpe = (spy_ann_return - rf_annual) / spy_ann_vol

summary_data.append({
    "Stock": "SPY (Market)",
    "Annualized Return": f"{spy_ann_return*100:.2f}%",
    "Annualized Volatility": f"{spy_ann_vol*100:.2f}%",
    "Beta (Systematic Risk)": 1.0,
    "Annualized Alpha": "0.00%",
    "Sharpe Ratio": round(spy_sharpe, 3),
    "Treynor Ratio": round((spy_ann_return - rf_annual) / 1.0, 3)
})

df_summary = pd.DataFrame(summary_data).set_index("Stock")
df_summary

## 6. Security Characteristic Line (SCL) Visualization

The Security Characteristic Line (SCL) plots the excess returns of a stock against the excess returns of the market. The slope is Beta and the intercept is Alpha.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for idx, stock in enumerate(stocks):
    sns.regplot(
        x=excess_returns['SPY'] * 100,  # in %
        y=excess_returns[stock] * 100,  # in %
        ax=axes[idx],
        color='C' + str(idx),
        scatter_kws={'alpha': 0.4, 's': 15},
        line_kws={'color': 'red', 'label': f"Beta: {results[stock]['beta']:.2f}"}
    )
    axes[idx].set_title(f"Security Characteristic Line (SCL): {stock}", fontsize=13, weight='bold')
    axes[idx].set_xlabel("Market Excess Returns (%)")
    axes[idx].set_ylabel(f"{stock} Excess Returns (%)")
    axes[idx].legend(fontsize=11)

plt.tight_layout()
plt.show()